In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import aioboto3
import asyncio

profile_name = "odin-cdk"
# session = aioboto3.Session(profile_name=profile_name, region_name="eu-north-1")
#credentials = session.get_credentials()

In [23]:
import json
from datetime import datetime
from typing import List, TypedDict

from botocore.config import Config

semaphore = asyncio.Semaphore(10)

STATE_MACHINE = (
    "arn:aws:states:eu-north-1:991049544436:stateMachine:OdinSMROdincalStateMachine"
)


class State(TypedDict):
    arn: str
    job: str
    status: str
    date: datetime | None
    error: str | None
    cause: str| None

History = List[State]

BOTO_CONFIG = Config(
    retries={"max_attempts": 10, "mode": "adaptive"}
)

In [24]:
from json import JSONDecodeError


async def fetch_execution_details(sfn_client, execution):
    """Fetch details of a single execution asynchronously."""
    async with semaphore:
        info = await sfn_client.describe_execution(
            executionArn=execution["executionArn"]
        )
        input_data = json.loads(info["input"])
        error = info["error"]
        if error == "InternalError":
            try:
                errmsg = json.loads(info.get("cause", "{}"))
            except JSONDecodeError:
                errmsg = {}
            if "non-replication superuser" in errmsg["errorMessage"]:
                print(f"redrive {info['executionArn']}, {errmsg['errorMessage']}")
                try:
                    await sfn_client.redrive_execution(
                        executionArn=info["executionArn"]
                    )
                except sfn_client.exceptions.ExecutionNotRedrivable as e:
                    try:
                        new_execution = await sfn_client.start_execution(
                            stateMachineArn=info["stateMachineArn"],
                            input=json.dumps(input_data),
                        )
                        print(f"New execution started: {new_execution['executionArn']}")
                    except Exception as start_error:
                        print(f"Failed to restart execution {info['executionArn']}: {start_error}")

        return State(
            arn=info["executionArn"],
            job=input_data.get("name", "Unknown"),
            status=execution["status"],
            date=execution.get("stopDate"),
            error=info.get("error"),
            cause=error,
        )

In [25]:
async def get_history(nmax: int) -> List[State]:
    """Retrieve failed executions asynchronously for maximum performance."""
    executions = []

    session = aioboto3.Session(profile_name=profile_name, region_name="eu-north-1")
    async with session.client("stepfunctions", config=BOTO_CONFIG) as sfn_client:
        paginator = sfn_client.get_paginator("list_executions")
        async for response in paginator.paginate(
            stateMachineArn=STATE_MACHINE,
            statusFilter="FAILED",
            PaginationConfig={"MaxItems": nmax, "PageSize": 100},
        ):
            executions.extend(response["executions"])

        # Fetch execution details concurrently
        tasks = [fetch_execution_details(sfn_client, exec) for exec in executions]
        history = await asyncio.gather(*tasks)
    return history

In [26]:
async def main():
    history = await get_history(1500)  # Fetch history for 50 failed executions


In [27]:
await main()
#c6da7ad9-38ae-42f4-a4c7-4f50cd40f128

redrive arn:aws:states:eu-north-1:991049544436:execution:OdinSMROdincalStateMachine:309ba5f7-0a6f-492c-a9a1-edc7dbb6d551, FATAL:  remaining connection slots are reserved for non-replication superuser connections

redrive arn:aws:states:eu-north-1:991049544436:execution:OdinSMROdincalStateMachine:ca0e76a8-5c6d-4262-97dc-876e8aa1a7ae, FATAL:  remaining connection slots are reserved for non-replication superuser connections

New execution started: arn:aws:states:eu-north-1:991049544436:execution:OdinSMROdincalStateMachine:f108e9ac-24c5-4855-8787-9eb2230d0038
New execution started: arn:aws:states:eu-north-1:991049544436:execution:OdinSMROdincalStateMachine:0870569e-581a-478f-b71a-b02790652725
redrive arn:aws:states:eu-north-1:991049544436:execution:OdinSMROdincalStateMachine:9042da95-63b2-443b-903a-5e60b1f95f52, FATAL:  remaining connection slots are reserved for non-replication superuser connections

New execution started: arn:aws:states:eu-north-1:991049544436:execution:OdinSMROdincalSta